In [ ]:
import gzip
import json
import re
import pandas as pd
from lxml import etree
from pathlib import Path
from difflib import SequenceMatcher
from collections import Counter
import os
import sys

def normalize_for_matching(text):
    """Normalize text for comparison: lowercase, remove non-alphanumeric, collapse whitespace."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def build_word_frequency_corpus(corpus_dir: Path) -> Counter:
    """
    Builds a word frequency counter from a directory of text files,
    including gzipped JSONs (like VILA) and plain text.
    Crucially, applies basic ligature fixes to corpus content before counting words
    to ensure full words like "unfortunately" are counted even if they come from
    hyphenated/ligated forms in the raw corpus source.
    """
    word_counts = Counter()
    if not corpus_dir.exists():
        print(f"Warning: Corpus path '{corpus_dir}' does not exist. Cannot build frequency corpus.", file=sys.stderr)
        return word_counts

    print(f"Building word frequency corpus from '{corpus_dir}'...")
    file_count = 0
    for root, _, files in os.walk(corpus_dir):
        for file_name in files:
            file_path = Path(root) / file_name
            content = ""
            try:
                if file_path.suffix == '.gz':
                    if 'json' in file_path.name.lower():
                        with gzip.open(file_path, 'rt', encoding='utf-8') as f:
                            data = json.load(f)
                            content = data.get("symbols", "")
                    else:
                        with gzip.open(file_path, 'rt', encoding='utf-8') as f:
                            content = f.read()
                elif file_path.suffix == '.json':
                    with open(file_path, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                        content = data.get("symbols", "")
                else:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        content = f.read()

                # --- NEW: Apply basic text cleaning to corpus content BEFORE counting ---
                # This ensures words like "unfortunately" get into the corpus
                # even if they were originally "Unfor-\ntunately" or had ligatures.
                temp_cleaned_content = content.replace('ï¬', 'ffi')
                temp_cleaned_content = temp_cleaned_content.replace('ï¬‚', 'ffl')
                temp_cleaned_content = temp_cleaned_content.replace('â€¢', '•')

                # A more aggressive dehyphenation for corpus building (might not be needed if this fix works)
                # This is a simpler dehyphenation just for populating the corpus, not the main dehyphenator.
                temp_cleaned_content = re.sub(r'([a-zA-Z]+)-\s*\n\s*([a-zA-Z]+)', r'\1\2', temp_cleaned_content)


                words = re.findall(r'\b[a-z]+\b', temp_cleaned_content.lower()) # Count lowercased words
                word_counts.update(words)
                file_count += 1
            except Exception as e:
                print(f"Error reading corpus file '{file_path}' for frequency building: {e}", file=sys.stderr)
    print(f"Finished building corpus from {file_count} files. Loaded {len(word_counts)} unique words.")
    return word_counts

def dehyphenate_text_with_corpus(text: str, word_frequencies: Counter) -> str:
    """
    Dehyphenates text by checking word frequencies for hyphenated words at line breaks.
    Attempts to preserve original casing, especially for sentence-starting words.
    Includes a fallback for zero frequencies if the word looks like a clear dehyphenation candidate.
    """
    if not isinstance(text, str):
        return ""

    if not word_frequencies:
        print("Warning: Word frequency corpus not loaded. Dehyphenation will rely on basic heuristics.", file=sys.stderr)
        # If no corpus, we'll try a simpler dehyphenation
        return re.sub(r'([a-zA-Z]+)-\s*\n\s*([a-zA-Z]+)', r'\1\2', text)


    lines = text.split('\n')
    processed_lines = []
    i = 0
    while i < len(lines):
        current_line = lines[i]

        if i + 1 < len(lines):
            next_line = lines[i+1]

            match_candidates = list(re.finditer(r'([a-zA-Z]+)-(\s*)$', current_line))

            if match_candidates:
                match = match_candidates[-1]
                original_first_part_full = match.group(1)
                trailing_spaces = match.group(2)

                next_word_match = re.match(r'^\s*([a-zA-Z]+)\b', next_line)

                if next_word_match:
                    original_second_part_full = next_word_match.group(1)

                    candidate_dehyphenated_lower = (original_first_part_full + original_second_part_full).lower()
                    candidate_hyphenated_retained_lower = (original_first_part_full + '-' + original_second_part_full).lower()

                    freq_dehyphenated = word_frequencies.get(candidate_dehyphenated_lower, 0)
                    freq_hyphenated_retained = word_frequencies.get(candidate_hyphenated_retained_lower, 0)

                    should_dehyphenate = False
                    if (freq_dehyphenated > 0 and freq_hyphenated_retained == 0) or \
                       (freq_dehyphenated > freq_hyphenated_retained * 2 and freq_dehyphenated > 0):
                        should_dehyphenate = True
                    elif freq_dehyphenated == 0 and freq_hyphenated_retained == 0:
                        should_dehyphenate = True

                    if should_dehyphenate:
                        first_char_capitalized = original_first_part_full[0].isupper()
                        combined_word = original_first_part_full + original_second_part_full

                        if first_char_capitalized:
                            resolved_word = combined_word[0].upper() + combined_word[1:]
                        else:
                            resolved_word = combined_word

                        current_line_modified = current_line[:match.start()] + resolved_word + trailing_spaces
                        next_line_modified = re.sub(r'^\s*' + re.escape(original_second_part_full) + r'\b', '', next_line, 1)

                        processed_lines.append(current_line_modified)
                        lines[i+1] = next_line_modified
                        i += 1
                        continue
                else:
                    pass # Next line does not start with an alphabetic word after hyphen.
            else:
                pass # No hyphenated word at the end of the current line

        processed_lines.append(current_line)
        i += 1

    return '\n'.join(processed_lines)

def extract_vila_sections(json_content):
    """
    Extracts sections from VILA JSON content using a more flexible regex pattern.
    It now handles numbers with newlines before headings and a wider range of
    characters in headings, including common punctuation.
    """
    if not isinstance(json_content, str):
        return []
    sections = []

    # Updated pattern to be more flexible with heading text and allow newlines
    # between a section number and the heading text itself.
    pattern = re.compile(
        r'^\s*(?P<number>\d+(?:\.\d+)*\s*(?:\n\s*)?)?' # Optional number (e.g., "3.1 "), possibly with newline/spaces after
        r'(?P<heading_text>[\w\s.,:;\'"()_/-]+?)' # Flexible for heading content: word chars, space, and common punctuation
                                                  # `+?` for non-greedy match.
        r'\s*\n+' # Heading is followed by one or more newlines (and optional whitespace)
        r'(?P<content>.*?)' # Non-greedy match for content
        # Lookahead: next section starts with optional number, then more flexible heading text, then newlines, or end of string.
        r'(?=\n^\s*(?:\d+(?:\.\d+)*\s*(?:\n\s*)?)?[\w\s.,:;\'"()_/-]+?\s*\n+|\Z)',
        re.MULTILINE | re.DOTALL | re.IGNORECASE # IGNORECASE allows matching "Abstract" as well as "ABSTRACT"
    )

    for match in pattern.finditer(json_content):
        heading = match.group('heading_text').strip()
        text = match.group('content').strip()
        num_part_full = match.group('number')

        level = 0
        if num_part_full:
            # Clean the num_part to just the number string for level calculation
            num_part = re.sub(r'\s*\n\s*', '', num_part_full).strip()
            if num_part:
                level = num_part.count('.') + 1

        if not heading: # Ensure we don't add empty headings
            continue

        sections.append((level, heading, text))
    return sections


def extract_grobid_headings(file_path):
    if not isinstance(file_path, Path):
        file_path = Path(file_path)
    try:
        try:
            with open(file_path, 'rb') as f:
                tree = etree.parse(f)
        except etree.XMLSyntaxError:
            with gzip.open(file_path, 'rb') as f:
                tree = etree.parse(f)
        ns = {'tei': 'http://www.tei-c.org/ns/1.0'}
        headings = []
        for head_element in tree.xpath('//tei:body//tei:head[@n]', namespaces=ns):
            n_value = head_element.get('n')
            head_text_elements = head_element.xpath('./text()', namespaces=ns)
            head_text = head_text_elements[0].strip() if head_text_elements else ''
            if not head_text:
                continue
            parts = n_value.split('.')
            level = len(parts)
            if level == 1:
                headings.append({
                    'level': level,
                    'n_value': n_value,
                    'text': head_text,
                    'subheadings': []
                })
            else:
                parent_n = '.'.join(parts[:-1])
                found_parent = False
                # Try to find the immediate parent
                for main_heading in headings:
                    if main_heading['n_value'] == parent_n:
                        main_heading['subheadings'].append({
                            'level': level,
                            'n_value': n_value,
                            'text': head_text
                        })
                        found_parent = True
                        break
                if not found_parent:
                    # If parent not found, add it as a new top-level heading.
                    # This might happen if a parent heading was not marked with @n or was malformed.
                    # Or if Grobid output isn't strictly hierarchical.
                    headings.append({
                        'level': level,
                        'n_value': n_value,
                        'text': head_text,
                        'subheadings': []
                    })
        return headings
    except Exception as e:
        print(f"Error processing GROBID file '{file_path}': {e}", file=sys.stderr)
        return []

def process_files_to_csv(vila_dir, grobid_dir, output_csv, min_score=0.6, corpus_for_dehyphenation=None):
    vila_dir = Path(vila_dir)
    grobid_dir = Path(grobid_dir)
    global_word_frequencies = Counter()
    if corpus_for_dehyphenation:
        global_word_frequencies = build_word_frequency_corpus(Path(corpus_for_dehyphenation))
        print(f"Successfully loaded {len(global_word_frequencies)} unique words for dehyphenation.")
    else:
        print("No corpus path provided for dehyphenation. Dehyphenation will be skipped.", file=sys.stderr)

    vila_files = sorted(list(vila_dir.glob("*.json.gz")))
    grobid_files = sorted(list(grobid_dir.glob("*.tei*")))

    grobid_map = {}
    for g_path in grobid_files:
        base_name = g_path.name.split('.grobid')[0]
        grobid_map[base_name] = g_path

    all_matched_records = []
    processed_paper_ids = set()

    print(f"\nStarting to process {len(vila_files)} VILA files...")
    for v_path in vila_files:
        paper_id = v_path.stem.split('.')[0]

        if paper_id in processed_paper_ids:
            print(f"Skipping already processed paper: {paper_id}", file=sys.stderr)
            continue

        print(f"Processing paper: {paper_id}")

        if paper_id not in grobid_map:
            print(f"Warning: No matching GROBID file found for '{paper_id}'. Skipping.", file=sys.stderr)
            continue

        g_path = grobid_map[paper_id]

        try:
            with gzip.open(v_path, 'rt', encoding='utf-8') as f:
                vila_data = json.load(f)
                vila_raw_text = vila_data.get("symbols", "")

                vila_processed_text = vila_raw_text.replace('ï¬', 'ffi')
                vila_processed_text = vila_processed_text.replace('ï¬‚', 'ffl')
                vila_processed_text = vila_processed_text.replace('â€¢', '•')

                if global_word_frequencies:
                    print(f"--- Dehyphenating for paper {paper_id} ---")
                    vila_processed_text = dehyphenate_text_with_corpus(vila_processed_text, global_word_frequencies)
                    print(f"--- Dehyphenation complete for paper {paper_id} ---")

                vila_sections = extract_vila_sections(vila_processed_text)
                if not vila_sections:
                    print(f"Warning: No VILA sections extracted for {paper_id}. Skipping matching.", file=sys.stderr)
                    continue

            grobid_headings = extract_grobid_headings(g_path)
            if not grobid_headings:
                print(f"Warning: No GROBID headings extracted for {paper_id}. Skipping matching.", file=sys.stderr)
                continue

            for grobid_main_heading in grobid_headings:
                grobid_heading_text = grobid_main_heading['text']
                normalized_grobid_heading = normalize_for_matching(grobid_heading_text)

                best_vila_match = None
                highest_match_score = 0

                for vila_level, vila_heading_text, vila_section_content in vila_sections:
                    normalized_vila_heading = normalize_for_matching(vila_heading_text)

                    match_score = SequenceMatcher(
                        None,
                        normalized_vila_heading,
                        normalized_grobid_heading
                    ).ratio()

                    if match_score > highest_match_score and match_score >= min_score:
                        highest_match_score = match_score
                        best_vila_match = (vila_heading_text, vila_section_content)

                subheadings_str = "; ".join(
                    f"{sub['n_value']}: {sub['text']}"
                    for sub in grobid_main_heading['subheadings']
                )

                if best_vila_match:
                    all_matched_records.append({
                        "paper_id": paper_id,
                        "section_name_grobid": grobid_heading_text,
                        "section_content_vila": best_vila_match[1],
                        "subheadings_grobid": subheadings_str
                    })

            processed_paper_ids.add(paper_id)

        except Exception as e:
            print(f"Critical error processing paper '{paper_id}': {e}", file=sys.stderr)

    if all_matched_records:
        df = pd.DataFrame(all_matched_records)
        df.to_csv(output_csv, index=False, encoding='utf-8')
        print(f"\nProcessing complete! Results saved to '{output_csv}'")
        print(f"Total unique papers with matched sections: {len(processed_paper_ids)}")
        print(f"Total sections matched and exported: {len(df)}")
    else:
        print("\nNo matching data found to export after processing all files.", file=sys.stderr)

if __name__ == "__main__":
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("Google Drive mounted successfully.")
    except ImportError:
        print("Not running in Google Colab environment. Skipping Google Drive mount.")
    except Exception as e:
        print(f"Error mounting Google Drive: {e}", file=sys.stderr)

    VILA_INPUT_DIR = "/content/drive/MyDrive/vila_test"
    GROBID_INPUT_DIR = "/content/drive/MyDrive/grobid_test"
    OUTPUT_CSV_FILE = "matched_sections_cleaned_final.csv"
    MATCHING_MIN_SCORE = 0.6
    CORPUS_FOR_DEHYPHENATION = VILA_INPUT_DIR

    print("\n--- Starting Text Processing Script ---")
    process_files_to_csv(
        vila_dir=VILA_INPUT_DIR,
        grobid_dir=GROBID_INPUT_DIR,
        output_csv=OUTPUT_CSV_FILE,
        min_score=MATCHING_MIN_SCORE,
        corpus_for_dehyphenation=CORPUS_FOR_DEHYPHENATION
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully.

--- Starting Text Processing Script ---
Building word frequency corpus from '/content/drive/MyDrive/vila_test'...
Finished building corpus from 3 files. Loaded 3491 unique words.
Successfully loaded 3491 unique words for dehyphenation.

Starting to process 3 VILA files...
Processing paper: -4hMlsXK4st
--- Dehyphenating for paper -4hMlsXK4st ---
--- Dehyphenation complete for paper -4hMlsXK4st ---
Processing paper: _0kaDkv3dVf
--- Dehyphenating for paper _0kaDkv3dVf ---
--- Dehyphenation complete for paper _0kaDkv3dVf ---
Processing paper: cO1IH43yUF
--- Dehyphenating for paper cO1IH43yUF ---
--- Dehyphenation complete for paper cO1IH43yUF ---

Processing complete! Results saved to 'matched_sections_cleaned_final.csv'
Total unique papers with matched sections: 3
Total sections matched and exported: 20


In [2]:
import gzip
import json
import re
import pandas as pd
from lxml import etree
from pathlib import Path
from difflib import SequenceMatcher
from collections import Counter
import os
import sys # Re-introduce for stderr, as logging errors is important
import csv

class DocumentSectionMatcher:
    def __init__(self, vila_dir: str, grobid_dir: str, output_csv: str, min_score: float = 0.6, corpus_for_dehyphenation: str = None):
        self.vila_dir = Path(vila_dir)
        self.grobid_dir = Path(grobid_dir)
        self.output_csv = output_csv
        self.min_score = min_score
        self.corpus_for_dehyphenation = Path(corpus_for_dehyphenation) if corpus_for_dehyphenation else None
        self.global_word_frequencies = Counter()
        self.processed_paper_ids = set()
        self.all_matched_records = []
        print(f"Matcher initialized with VILA: {self.vila_dir}, GROBID: {self.grobid_dir}, Output: {self.output_csv}")
        print(f"Min match score: {self.min_score}, Dehyphenation corpus: {self.corpus_for_dehyphenation}")

    def _normalize_for_matching(self, text: str) -> str:
        if not isinstance(text, str):
            return ""
        text = text.lower()
        text = re.sub(r'[^\w\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def _build_word_frequency_corpus(self) -> Counter:
        print("Building word frequency corpus for dehyphenation...")
        word_counts = Counter()
        if not self.corpus_for_dehyphenation or not self.corpus_for_dehyphenation.exists():
            print("Dehyphenation corpus path invalid or not provided. Skipping corpus build.")
            return word_counts

        for root, _, files in os.walk(self.corpus_for_dehyphenation):
            for file_name in files:
                file_path = Path(root) / file_name
                content = ""
                try:
                    # Minimal try-except for reading each file
                    if file_path.suffix == '.gz':
                        if 'json' in file_path.name.lower():
                            with gzip.open(file_path, 'rt', encoding='utf-8') as f:
                                data = json.load(f)
                                content = data.get("symbols", "")
                        else:
                            with gzip.open(file_path, 'rt', encoding='utf-8') as f:
                                content = f.read()
                    elif file_path.suffix == '.json':
                        with open(file_path, 'r', encoding='utf-8') as f:
                            data = json.load(f)
                            content = data.get("symbols", "")
                    else:
                        with open(file_path, 'r', encoding='utf-8') as f:
                            content = f.read()

                    # These operations are less likely to fail critically after content is read,
                    # but if they did, the error would still be caught by the outer try
                    temp_cleaned_content = content.replace('ï¬', 'ffi').replace('ï¬‚', 'ffl').replace('â€¢', '•')
                    temp_cleaned_content = re.sub(r'([a-zA-Z]+)-\s*\n\s*([a-zA-Z]+)', r'\1\2', temp_cleaned_content)

                    words = re.findall(r'\b[a-z]+\b', temp_cleaned_content.lower())
                    word_counts.update(words)
                except (IOError, json.JSONDecodeError, UnicodeDecodeError, OSError) as e:
                    # Catch specific file-related errors and report for this file, then continue
                    print(f"    Error reading or parsing corpus file {file_path}: {e}", file=sys.stderr)
                except Exception as e:
                    # Catch any other unexpected errors during content processing for this file
                    print(f"    An unexpected error occurred processing content of {file_path}: {e}", file=sys.stderr)
        print(f"Corpus built with {len(word_counts)} unique words.")
        return word_counts

    def _dehyphenate_text_with_corpus(self, text: str) -> str:
        if not isinstance(text, str):
            return ""
        if not self.global_word_frequencies:
            # If no corpus, just apply basic dehyphenation
            return re.sub(r'([a-zA-Z]+)-\s*\n\s*([a-zA-Z]+)', r'\1\2', text)

        lines = text.split('\n')
        processed_lines = []
        i = 0
        while i < len(lines):
            current_line = lines[i]
            if i + 1 < len(lines):
                next_line = lines[i+1]
                match_candidates = list(re.finditer(r'([a-zA-Z]+)-(\s*)$', current_line))
                if match_candidates:
                    match = match_candidates[-1]
                    original_first_part_full = match.group(1)
                    trailing_spaces = match.group(2)
                    next_word_match = re.match(r'^\s*([a-zA-Z]+)\b', next_line)

                    if next_word_match:
                        original_second_part_full = next_word_match.group(1)
                        candidate_dehyphenated_lower = (original_first_part_full + original_second_part_full).lower()
                        candidate_hyphenated_retained_lower = (original_first_part_full + '-' + original_second_part_full).lower()
                        freq_dehyphenated = self.global_word_frequencies.get(candidate_dehyphenated_lower, 0)
                        freq_hyphenated_retained = self.global_word_frequencies.get(candidate_hyphenated_retained_lower, 0)

                        should_dehyphenate = False
                        if (freq_dehyphenated > 0 and freq_hyphenated_retained == 0) or \
                           (freq_dehyphenated > freq_hyphenated_retained * 2 and freq_dehyphenated > 0):
                            should_dehyphenate = True
                        elif freq_dehyphenated == 0 and freq_hyphenated_retained == 0:
                            # If neither is in corpus, assume dehyphenation is safe as a default heuristic
                            should_dehyphenate = True

                        if should_dehyphenate:
                            first_char_capitalized = original_first_part_full[0].isupper()
                            combined_word = original_first_part_full + original_second_part_full
                            resolved_word = combined_word[0].upper() + combined_word[1:] if first_char_capitalized else combined_word
                            current_line_modified = current_line[:match.start()] + resolved_word + trailing_spaces
                            next_line_modified = re.sub(r'^\s*' + re.escape(original_second_part_full) + r'\b', '', next_line, 1)
                            processed_lines.append(current_line_modified)
                            lines[i+1] = next_line_modified
                            i += 1
                            continue # Move to the next line immediately as two lines were consumed
            processed_lines.append(current_line)
            i += 1
        return '\n'.join(processed_lines)

    def _extract_vila_sections(self, json_content: str) -> list:
        if not isinstance(json_content, str):
            return []
        sections = []
        # The regex pattern itself is robust; the content should already be a string here.
        # No specific try-except needed around the regex matching itself.
        pattern = re.compile(
            r'^\s*(?P<number>\d+(?:\.\d+)*\s*(?:\n\s*)?)?'
            r'(?P<heading_text>[\w\s.,:;\'"()_/-]+?)'
            r'\s*\n+'
            r'(?P<content>.*?)'
            r'(?=\n^\s*(?:\d+(?:\.\d+)*\s*(?:\n\s*)?)?[\w\s.,:;\'"()_/-]+?\s*\n+|\Z)',
            re.MULTILINE | re.DOTALL | re.IGNORECASE
        )
        for match in pattern.finditer(json_content):
            heading = match.group('heading_text').strip()
            text = match.group('content').strip()
            num_part_full = match.group('number')
            level = 0
            if num_part_full:
                num_part = re.sub(r'\s*\n\s*', '', num_part_full).strip()
                if num_part:
                    level = num_part.count('.') + 1
            if heading:
                sections.append((level, heading, text))
        return sections

    def _extract_grobid_headings(self, file_path: Path) -> list:
        """
        Extracts GROBID headings from an XML or gzipped XML file.
        Uses sequential try-except blocks to avoid nesting.
        """
        tree = None
        data_read = None

        # Attempt 1: Read as a regular file (for .tei.xml or similar)
        try:
            with open(file_path, 'rb') as f:
                data_read = f.read()
            tree = etree.fromstring(data_read) # Use fromstring for parsing raw bytes
        except (IOError, OSError) as e:
            # File not found or inaccessible, this is a distinct failure.
            print(f"    Error reading file '{file_path}' (as regular file): {e}", file=sys.stderr)
            return [] # Critical failure to read, stop here.
        except etree.XMLSyntaxError as e:
            # It's an XML syntax error, means it's likely not plain XML,
            # or it's malformed XML that's not gzipped.
            # We'll now try it as gzipped, but log this attempt.
            print(f"    '{file_path}' is not valid plain XML or is malformed: {e}. Trying as gzip...", file=sys.stderr)
            # Do NOT return here, proceed to gzip attempt

        if tree is None: # Only proceed to gzip attempt if first attempt failed to parse XML
            # Attempt 2: Read as a gzipped file (for .tei.xml.gz)
            try:
                with gzip.open(file_path, 'rb') as f:
                    data_read = f.read()
                tree = etree.fromstring(data_read)
            except (IOError, OSError, gzip.BadGzipFile) as e:
                # Catch issues specific to gzip and reading it
                print(f"    Error reading/parsing '{file_path}' as gzipped XML: {e}", file=sys.stderr)
                return [] # If both failed, then there's no tree.
            except etree.XMLSyntaxError as e:
                # Gzipped but still bad XML syntax
                print(f"    '{file_path}' is gzipped but XML content is malformed: {e}", file=sys.stderr)
                return []

        if tree is None: # If after all attempts, no tree was successfully parsed
            print(f"    Failed to parse XML from '{file_path}' after all attempts.", file=sys.stderr)
            return []

        # Now, process the successfully obtained XML tree to extract headings
        try:
            ns = {'tei': 'http://www.tei-c.org/ns/1.0'}
            raw_headings_data = []
            for head_element in tree.xpath('//tei:body//tei:head[@n]', namespaces=ns):
                n_value = head_element.get('n')
                head_text_elements = head_element.xpath('./text()', namespaces=ns)
                head_text = head_text_elements[0].strip() if head_text_elements else ''
                if head_text:
                    raw_headings_data.append({
                        'level': len(n_value.split('.')),
                        'n_value': n_value,
                        'text': head_text
                    })

            structured_headings = []
            heading_map = {}

            for h_data in raw_headings_data:
                current_heading = {
                    'level': h_data['level'],
                    'n_value': h_data['n_value'],
                    'text': h_data['text'],
                    'subheadings': []
                }
                heading_map[h_data['n_value']] = current_heading

                if h_data['level'] == 1:
                    structured_headings.append(current_heading)
                else:
                    parent_n = '.'.join(h_data['n_value'].split('.')[:-1])
                    if parent_n in heading_map:
                        heading_map[parent_n]['subheadings'].append(current_heading)
                    else:
                        structured_headings.append(current_heading) # Fallback for orphans
            return structured_headings
        except Exception as e:
            # This catch is for errors during the XPath query or data structuring,
            # *after* the file has been successfully parsed into a tree.
            print(f"    Error processing GROBID XML structure in '{file_path.name}': {e}", file=sys.stderr)
            return []

    def _add_processed_paper_id(self, paper_id: str):
        """Adds a paper_id to the set of processed IDs."""
        self.processed_paper_ids.add(paper_id)
        print(f"    --> Marked {paper_id} as processed.")

    def save_results_to_csv(self):
        """Saves the accumulated matched records to the specified CSV file."""
        print(f"Attempting to save {len(self.all_matched_records)} records to {self.output_csv}...")
        if self.all_matched_records:
            # --- Internal Data Structure Check (for debugging, can be removed in production) ---
            print("\n--- Internal Data Structure Check (first 3 records from self.all_matched_records) ---")
            for i, record in enumerate(self.all_matched_records[:3]):
                print(f"Record {i}:")
                print(f"    paper_id: '{record.get('paper_id', 'MISSING_ID')}' (type: {type(record.get('paper_id'))})")
                print(f"    section_name_grobid: '{record.get('section_name_grobid', '')[:50]}{'...' if len(record.get('section_name_grobid', '')) > 50 else ''}'")
                print(f"    section_content_vila (length: {len(record.get('section_content_vila', ''))}): {repr(record.get('section_content_vila', '')[:100])}")
                print(f"    subheadings_grobid: '{record.get('subheadings_grobid', '')[:50]}{'...' if len(record.get('subheadings_grobid', '')) > 50 else ''}'")
            print("-----------------------------------------------------")

            df = pd.DataFrame(self.all_matched_records)
            # Ensure consistent column order
            df = df[['paper_id', 'section_name_grobid', 'section_content_vila', 'subheadings_grobid']]

            # --- Pandas DataFrame Head (before saving to CSV) ---
            print("\n--- Pandas DataFrame Head (before saving to CSV) ---")
            print(df.head().to_string())
            print("----------------------------------------------------")

            # Save to CSV with proper quoting
            try:
                df.to_csv(self.output_csv, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL, sep=',')
                print(f"\nResults successfully saved to {self.output_csv}")
            except IOError as e:
                print(f"Error saving results to '{self.output_csv}': {e}", file=sys.stderr)
                return # Stop if saving fails

            # --- Verifying content of the first few lines of the *saved* CSV file (reading raw lines) ---
            print(f"\n--- Verifying content of the first few lines of {self.output_csv} (reading raw lines) ---")
            try:
                with open(self.output_csv, 'r', encoding='utf-8') as f:
                    for _ in range(5): # Read first 5 lines directly from the file
                        line = f.readline().strip()
                        if line:
                            print(line)
            except Exception as e:
                print(f"Error reading back CSV file for verification: {e}", file=sys.stderr)
            print(f"------------------------------------------------------------------")

        else:
            print("No matched records to save. Output CSV will not be created or will be empty.")

    def verify_saved_paper_ids(self):
        """
        Loads the saved CSV and prints the paper_id and its length for each row.
        """
        print(f"\n--- Verifying paper_id lengths from the saved CSV file: {self.output_csv} ---")
        try:
            df_saved = pd.read_csv(self.output_csv, encoding='utf-8', sep=',')
            if 'paper_id' in df_saved.columns:
                print(f"Total records loaded for verification: {len(df_saved)}")
                for i, paper_id in enumerate(df_saved['paper_id']):
                    pid_str = str(paper_id)
                    print(f"Row {i+1}: Paper ID: '{pid_str}', Length: {len(pid_str)}")
                    # Optionally, add a check for unusually long IDs
                    if len(pid_str) > 20 and 'Appended VILA sub-section content' in pid_str: # Typical paper IDs are shorter than 20 chars
                        print(f"      WARNING: Unusually long Paper ID found, potentially containing extra content!")
            else:
                print("Error: 'paper_id' column not found in the saved CSV.", file=sys.stderr)
        except FileNotFoundError:
            print(f"Error: Saved CSV file not found at {self.output_csv}. Cannot verify.", file=sys.stderr)
        except Exception as e:
            print(f"An error occurred during verification: {e}", file=sys.stderr)
        print("----------------------------------------------------------")

    def run_processing(self):
        print("Starting document section matching process...")
        if self.corpus_for_dehyphenation:
            self.global_word_frequencies = self._build_word_frequency_corpus()

        vila_files = sorted(list(self.vila_dir.glob("*.json.gz")))
        grobid_files = sorted(list(self.grobid_dir.glob("*.tei*")))
        grobid_map = {g_path.name.split('.grobid')[0]: g_path for g_path in grobid_files}

        print(f"Found {len(vila_files)} VILA files and {len(grobid_files)} GROBID files.")

        for v_path in vila_files:
            paper_id = v_path.stem.split('.')[0]
            print(f"\nProcessing paper_id: {paper_id}")

            if paper_id in self.processed_paper_ids:
                print(f"    Skipping {paper_id} as it was already processed.")
                continue

            if paper_id not in grobid_map:
                print(f"    No corresponding GROBID file found for {paper_id}. Cannot match. Skipping.")
                self._add_processed_paper_id(paper_id)
                continue

            g_path = grobid_map[paper_id]
            print(f"    Found matching GROBID file: {g_path.name}")

            # Load and process VILA data - minimal try-except for file I/O and JSON parsing
            vila_raw_text = ""
            try:
                with gzip.open(v_path, 'rt', encoding='utf-8') as f:
                    vila_data = json.load(f)
                    vila_raw_text = vila_data.get("symbols", "")
                print(f"    Loaded VILA raw text for {paper_id} (length: {len(vila_raw_text)}).")
            except (IOError, json.JSONDecodeError, UnicodeDecodeError, OSError) as e:
                print(f"    Error reading or parsing VILA file '{v_path}': {e}. Skipping paper.", file=sys.stderr)
                self._add_processed_paper_id(paper_id)
                continue # Skip to next paper

            # Text cleaning and dehyphenation - these operations are generally robust
            # and don't require individual try-except unless specific regex/string ops fail.
            vila_processed_text = vila_raw_text.replace('ï¬', 'ffi').replace('ï¬‚', 'ffl').replace('â€¢', '•')
            if self.global_word_frequencies:
                vila_processed_text = self._dehyphenate_text_with_corpus(vila_processed_text)
                print(f"    Applied dehyphenation to VILA text.")

            # Extract VILA sections - this operation is also generally robust with a good regex
            vila_all_sections = self._extract_vila_sections(vila_processed_text)
            print(f"    Extracted {len(vila_all_sections)} sections from VILA for {paper_id}.")

            if not vila_all_sections:
                print(f"    No VILA sections found for {paper_id}. Cannot match. Skipping.")
                self._add_processed_paper_id(paper_id)
                continue

            # Load and process GROBID headings - _extract_grobid_headings handles its own errors
            grobid_headings = self._extract_grobid_headings(g_path)
            print(f"    Extracted {len([h for h in grobid_headings if h['level'] == 1])} top-level GROBID headings for {paper_id}.")
            if not grobid_headings:
                print(f"    No GROBID headings found for {paper_id}. Cannot match. Skipping.")
                self._add_processed_paper_id(paper_id)
                continue

            # Iterate through GROBID main headings for matching
            for grobid_main_heading in grobid_headings:
                if grobid_main_heading['level'] != 1:
                    continue

                print(f"\n    Attempting to match GROBID main heading: '{grobid_main_heading['n_value']}: {grobid_main_heading['text']}'")
                normalized_grobid_heading = self._normalize_for_matching(grobid_main_heading['text'])
                grobid_n_value = grobid_main_heading['n_value']

                best_vila_match_index = -1
                highest_match_score = 0

                for idx, (_, vila_heading_text, _) in enumerate(vila_all_sections):
                    normalized_vila_heading = self._normalize_for_matching(vila_heading_text)
                    match_score = SequenceMatcher(None, normalized_vila_heading, normalized_grobid_heading).ratio()
                    if match_score > highest_match_score:
                        highest_match_score = match_score
                        best_vila_match_index = idx

                record = {
                    "paper_id": paper_id,
                    "section_name_grobid": grobid_main_heading['text'],
                    "section_content_vila": "", # Default to empty string if no content
                    "subheadings_grobid": "; ".join(f"{sub['n_value']}: {sub['text']}" for sub in grobid_main_heading['subheadings'])
                }

                if best_vila_match_index != -1 and highest_match_score >= self.min_score:
                    print(f"    --> Best VILA match found for '{grobid_main_heading['text']}' with score: {highest_match_score:.2f}")

                    initial_vila_content = vila_all_sections[best_vila_match_index][2]
                    accumulated_vila_content = initial_vila_content

                    print(f"    Starting content accumulation from VILA section: '{vila_all_sections[best_vila_match_index][1]}'")

                    stop_vila_index = len(vila_all_sections)
                    next_grobid_level_1_heading_text = None

                    for gh in grobid_headings:
                        try:
                            # Safely convert to float for comparison; handles non-numeric 'n_value'
                            if gh['level'] == 1 and isinstance(gh['n_value'], str) and gh['n_value'].replace('.', '', 1).isdigit() and \
                               float(gh['n_value']) > float(grobid_n_value):
                                next_grobid_level_1_heading_text = gh['text']
                                print(f"        Next GROBID main heading found: '{gh['n_value']}: {gh['text']}'")
                                break
                        except ValueError:
                            # Handle cases where n_value might not be purely numeric (e.g., "APPENDIX A")
                            # We'll just skip these for determining numeric section boundaries.
                            pass

                    if next_grobid_level_1_heading_text:
                        normalized_next_grobid_heading = self._normalize_for_matching(next_grobid_level_1_heading_text)
                        for k in range(best_vila_match_index + 1, stop_vila_index): # Ensure we don't go past the determined stop index
                            current_vila_heading_text = vila_all_sections[k][1]
                            normalized_current_vila_heading = self._normalize_for_matching(current_vila_heading_text)
                            if SequenceMatcher(None, normalized_current_vila_heading, normalized_next_grobid_heading).ratio() >= self.min_score:
                                stop_vila_index = k
                                # print(f"    VILA content accumulation stopping at VILA section: '{vila_all_sections[k][1]}'")
                                break # Found the boundary, stop searching

                    for l in range(best_vila_match_index + 1, stop_vila_index):
                        # Append heading and content for sub-sections if they appear within the main GROBID section's bounds
                        accumulated_vila_content += "\n\n" + vila_all_sections[l][1] + "\n" + vila_all_sections[l][2]
                        # print(f"    Appended VILA sub-section content: '{vila_all_sections[l][1]}'")

                    # print(f"    Accumulated VILA content length: {len(accumulated_vila_content)} chars")

                    # Update the content field
                    record["section_content_vila"] = accumulated_vila_content
                    # print(f"    --> Record added for {paper_id}, GROBID section '{grobid_main_heading['text']}' (MATCHED).")
                else:
                    # print(f"    No strong VILA match found for GROBID heading: '{grobid_main_heading['text']}' (highest score: {highest_match_score:.2f}).")
                    print(f"    --> Record added for {paper_id}, GROBID section '{grobid_main_heading['text']}' (UNMATCHED).")

                # Add the completed record to the list for each GROBID main heading
                self.all_matched_records.append(record)

            # Mark the paper as fully processed
            self._add_processed_paper_id(paper_id)

        self.save_results_to_csv()


if __name__ == "__main__":
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        print("Not in Google Colab environment.")
    except Exception as e:
        print(f"Error mounting Google Drive: {e}", file=sys.stderr)

    # Configuration
    VILA_INPUT_DIR = "/content/drive/MyDrive/vila_test"
    GROBID_INPUT_DIR = "/content/drive/MyDrive/grobid_test"
    OUTPUT_CSV_FILE = "matched_sections_filename_clean.csv"
    MATCHING_MIN_SCORE = 0.6
    CORPUS_FOR_DEHYPHENATION = VILA_INPUT_DIR # Or a path to a more general corpus if available

    # Crucial directory existence checks before starting the main process
    if not Path(VILA_INPUT_DIR).exists():
        print(f"Error: VILA input directory not found: {VILA_INPUT_DIR}", file=sys.stderr)
        sys.exit(1) # Exit if critical input directories are missing
    if not Path(GROBID_INPUT_DIR).exists():
        print(f"Error: GROBID input directory not found: {GROBID_INPUT_DIR}", file=sys.stderr)
        sys.exit(1)

    matcher = DocumentSectionMatcher(
        vila_dir=VILA_INPUT_DIR,
        grobid_dir=GROBID_INPUT_DIR,
        output_csv=OUTPUT_CSV_FILE,
        min_score=MATCHING_MIN_SCORE,
        corpus_for_dehyphenation=CORPUS_FOR_DEHYPHENATION
    )
    matcher.run_processing()
    matcher.verify_saved_paper_ids()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Matcher initialized with VILA: /content/drive/MyDrive/vila_test, GROBID: /content/drive/MyDrive/grobid_test, Output: matched_sections_filename_clean.csv
Min match score: 0.6, Dehyphenation corpus: /content/drive/MyDrive/vila_test
Starting document section matching process...
Building word frequency corpus for dehyphenation...
Corpus built with 3491 unique words.
Found 3 VILA files and 3 GROBID files.

Processing paper_id: -4hMlsXK4st
    Found matching GROBID file: -4hMlsXK4st.grobid.tei.xml
    Loaded VILA raw text for -4hMlsXK4st (length: 52740).
    Applied dehyphenation to VILA text.
    Extracted 536 sections from VILA for -4hMlsXK4st.
    Extracted 4 top-level GROBID headings for -4hMlsXK4st.

    Attempting to match GROBID main heading: '1: INTRODUCTION'
    --> Best VILA match found for 'INTRODUCTION' with score: 0.96
    Starting content accumulation